# Generadores

Un **generador** es una función que, en lugar de calcular todos sus resultados y
devolverlos de golpe en una lista, **entrega un elemento cada vez que se lo piden**. Así
nunca tiene toda la colección en memoria: solo el elemento actual.

Se escriben con la palabra clave **`yield`**, y en este notebook vamos a ver qué problema
resuelven, cómo funcionan por dentro y cómo se usan.

## 1. El problema: una función que devuelve una lista lo calcula todo

Imagina una función que produce los cuadrados de los números del 1 al `n`:

In [12]:
def cuadrados_lista(n):
    resultado = []
    for i in range(1, n + 1):
        resultado.append(i * i)
    return resultado


print(cuadrados_lista(5))

[1, 4, 9, 16, 25]


Funciona, pero fíjate en lo que ocurre al llamarla: **construye la lista entera** antes de
devolver nada. Si pedimos `cuadrados_lista(10_000_000)` para quedarnos solo con los tres
primeros, Python calcula y guarda en memoria diez millones de números para tirar casi
todos.

El mismo problema aparece al leer un fichero enorme con `f.readlines()`, al pedir todas las
filas de una base de datos o al generar todas las combinaciones posibles de algo: pagamos
memoria y tiempo por elementos que quizá nunca lleguemos a usar.

## 2. La solución: un generador entrega un elemento en cada petición

Cambiamos dos cosas: quitamos la lista y sustituimos `return` por **`yield`**:

In [13]:
def cuadrados(n):
    for i in range(1, n + 1):
        yield i * i


for cuadrado in cuadrados(5):
    print(cuadrado)

1
4
9
16
25


El resultado es el mismo, pero la forma de trabajar es completamente distinta. `yield`
significa: *"entrega este valor y quédate parado aquí hasta que te pidan el siguiente"*.

Para verlo, vamos a pedir los elementos a mano. Cuando llamamos a una función con `yield`
**no se ejecuta su código**: solo obtenemos un objeto *generador*. Con `next()` le pedimos
el siguiente elemento, y entonces la función avanza hasta el siguiente `yield`:

In [14]:
def cuadrados_con_mensajes(n):
    for i in range(1, n + 1):
        print(f"   (calculando {i} * {i})")
        yield i * i


gen = cuadrados_con_mensajes(3)
print("Generador creado. Todavía no se ha calculado nada.")

print("Pido el primero...")
print(next(gen))

print("Pido el segundo...")
print(next(gen))

print("Pido el tercero...")
print(next(gen))

Generador creado. Todavía no se ha calculado nada.
Pido el primero...
   (calculando 1 * 1)
1
Pido el segundo...
   (calculando 2 * 2)
4
Pido el tercero...
   (calculando 3 * 3)
9


Observa el orden de los mensajes: cada cuadrado se calcula **justo cuando se pide**, no
antes. En ningún momento hay una lista con los tres; solo existe el valor que acabamos de
pedir.

Un `for` hace exactamente esto por nosotros: llama a `next()` una y otra vez hasta que el
generador se queda sin elementos. Cuando eso pasa, el generador lanza una excepción
especial, `StopIteration`, que es la señal que usa el `for` para terminar:

In [15]:
try:
    next(gen)
except StopIteration:
    print("No quedan más: el generador ha lanzado StopIteration")

No quedan más: el generador ha lanzado StopIteration


## 3. Un generador se agota

Un generador va siempre hacia delante. Una vez que ha entregado todos sus elementos está
**agotado**: si lo recorremos otra vez, no produce nada.

In [16]:
gen = cuadrados(3)

print("Primera vuelta:", list(gen))
print("Segunda vuelta:", list(gen))

Primera vuelta: [1, 4, 9]
Segunda vuelta: []


Esto es diferente de una lista, que se puede recorrer tantas veces como se quiera. Si
necesitas volver a recorrer los elementos, o creas un generador nuevo (llamando otra vez a
`cuadrados(3)`) o los guardas en una lista con `list(gen)` la primera vez.

Ten cuidado también con esto: `len(gen)` no funciona (un generador no sabe cuántos
elementos le quedan sin recorrerlos) y `gen[0]` tampoco (no hay índices, solo "el
siguiente").

## 4. La ventaja: solo se calcula lo que se pide

Volvamos al problema inicial. Con el generador, pedir los tres primeros cuadrados de diez
millones calcula **tres** cuadrados y para:

In [17]:
for cuadrado in cuadrados(10_000_000):
    if cuadrado > 10:
        break
    print(cuadrado)

1
4
9


Al hacer `break`, el resto del bucle del generador nunca llega a ejecutarse. Esto permite
incluso generadores **infinitos**: el que consume decide cuándo parar.

In [18]:
def numeros_pares():
    n = 0
    while True:       # nunca termina por sí solo
        yield n
        n += 2


for par in numeros_pares():
    if par > 6:
        break
    print(par)

0
2
4
6


Para "quedarme con los primeros `n`" hay una herramienta hecha: `itertools.islice`. Toma
los primeros `n` elementos de un generador **sin recorrer el resto**, y también es
perezosa (no pide nada hasta que alguien la recorre):

In [19]:
from itertools import islice

primeros = islice(numeros_pares(), 5)   # todavía no ha pedido ningún elemento
print(list(primeros))                   # ahora pide exactamente cinco

[0, 2, 4, 6, 8]


## 5. Expresiones generadoras

Ya conoces las *list comprehensions*, una forma compacta de construir una lista:

In [20]:
numeros = [3, 8, 1, 9, 4]

cuadrados_de_numeros = [n * n for n in numeros]
print(cuadrados_de_numeros)

[9, 64, 1, 81, 16]


Como toda lista, se construye **entera** en el momento: los cinco cuadrados se calculan y
se guardan aunque después solo vayamos a sumarlos.

Una **expresión generadora** es lo mismo pero con **paréntesis** en vez de corchetes. No
construye ninguna lista: crea un generador, exactamente como si hubiéramos escrito una
función con `yield` que recorre `numeros` y entrega `n * n` de uno en uno.

In [21]:
gen = (n * n for n in numeros)

print(gen)          # un generador: todavía no ha calculado nada
print(next(gen))    # ahora calcula el primero: 3 * 3
print(next(gen))    # y el segundo: 8 * 8

<generator object <genexpr> at 0x7d80c25acee0>
9
64


Se usan sobre todo cuando el resultado se lo pasamos **directamente a una función que
consume los elementos de uno en uno**, como `sum`, `max`, `min`, `any` o `all`. Esas
funciones solo necesitan recorrer los valores una vez, así que no tiene sentido guardarlos
antes en una lista. Cuando la expresión generadora es el único argumento, se puede
escribir sin el paréntesis extra:

In [22]:
print(sum(n * n for n in numeros))              # suma de los cuadrados, sin lista intermedia
print(max(n for n in numeros if n % 2 == 0))    # el mayor de los pares
print(any(n > 8 for n in numeros))              # ¿hay alguno mayor que 8?

171
8
True


Regla práctica: si solo vas a recorrer el resultado una vez, paréntesis; si necesitas
guardarlo, indexarlo o recorrerlo varias veces, corchetes.

## 6. Generadores dentro de una clase

Un método también puede ser un generador. Es útil cuando un método tiene que devolver
"una serie de cosas" y quien lo llama solo va a recorrerlas: se escribe con `yield` en vez
de construir y devolver una lista.

In [23]:
class Equipo:
    def __init__(self, jugadores):
        self.jugadores = jugadores

    def con_inicial(self, letra):
        # Generador: los jugadores cuyo nombre empieza por `letra`, de uno en uno
        for jugador in self.jugadores:
            if jugador.startswith(letra):
                yield jugador


equipo = Equipo(["Magnus", "Aryan", "Johan", "Aleksander"])

for jugador in equipo.con_inicial("A"):
    print(jugador)

Aryan
Aleksander


Fíjate en que `equipo.con_inicial("A")` devuelve un generador nuevo cada vez que se llama.
Por eso se puede recorrer tantas veces como se quiera: cada llamada empieza desde el
principio, aunque el generador de la llamada anterior esté agotado.

In [24]:
print(list(equipo.con_inicial("A")))
print(list(equipo.con_inicial("A")))   # otra llamada, otro generador nuevo

['Aryan', 'Aleksander']
['Aryan', 'Aleksander']


Un caso especial muy habitual es el método `__iter__`, el que Python llama cuando haces
`for x in objeto`. Si lo escribimos como generador, el objeto se puede recorrer con `for`
directamente:

In [25]:
class Equipo:
    def __init__(self, jugadores):
        self.jugadores = jugadores

    def __iter__(self):
        # Recorrer el equipo = recorrer sus jugadores por orden alfabético
        for jugador in sorted(self.jugadores):
            yield jugador


equipo = Equipo(["Magnus", "Aryan", "Johan"])

for jugador in equipo:
    print(jugador)

Aryan
Johan
Magnus


> 💡 `yield` no es la única manera de conseguir esto: también se puede escribir una clase
> con los métodos `__iter__` y `__next__` que lleve la cuenta de por dónde va (es lo que
> Python llama un *iterador*). Es más largo y más propenso a errores, así que en este
> curso usaremos siempre la forma con `yield`.